# RSNA Knee MRI — Huấn luyện trên Colab (A100 80GB)

Text-guided knowledge distillation: **huấn luyện** bằng ảnh + report,
**inference** chỉ cần ảnh + metadata series.

Notebook này chỉ *gọi* các hàm trong package `knee_mri`; toàn bộ logic nằm trong
`src/` nên chạy được cả ngoài notebook và có test bao phủ.

## Cách dùng

1. Đặt `MODE = 'pilot'` (cell 2) → chạy **toàn bộ** notebook từ trên xuống. Pilot dùng
   *toàn bộ 58 study gold* + một mẫu nhỏ study không nhãn, artifacts ghi riêng vào
   `artifacts/pilot/`. Cuối notebook có ước lượng thời gian cho bản đầy đủ.
2. Nếu pilot cho AUC > ngẫu nhiên và thời gian chấp nhận được → đổi `MODE = 'full'`,
   **Restart runtime**, chạy lại từ đầu.
3. Runtime bị ngắt giữa chừng? Chạy lại từ đầu: mọi bước precompute bỏ qua file đã
   có, còn huấn luyện tự tiếp tục từ `last.pt` (`resume=True`).

Drive cần có: `dataset/` (dữ liệu RSNA). `artifacts/` và `vendor/` sẽ tự được tạo.

In [ ]:
# 0. Bootstrap: lấy code, cài package, nạp cấu hình — chạy được cả Colab lẫn local.
#    Code luôn được clone/pull vào /content (Colab) rồi `pip install -e`, nên không còn
#    phụ thuộc vào việc bạn đã tự chép src/ lên Drive. Chỉ dataset/, artifacts/ và
#    vendor/ nằm trên Drive (bền vững giữa các lần runtime bị ngắt).
import logging, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hoangtung386/Att-kneenet.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Challenge_RNSA_KNEE')
    CODE = Path('/content/Att-kneenet')
    if (CODE / '.git').is_dir():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CODE)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[viz,eval]'], check=True)
else:
    CODE = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
    DRIVE_ROOT = CODE
sys.path.insert(0, str(CODE / 'src'))

assert (DRIVE_ROOT / 'dataset' / 'train.csv').is_file(), (
    f'Không thấy dataset tại {DRIVE_ROOT / "dataset"} — hãy đặt dataset RSNA vào đó.'
)
print('code   :', CODE)
print('dataset:', DRIVE_ROOT / 'dataset')

In [ ]:
# 1. Chế độ chạy — ĐỔI Ở ĐÂY
MODE = 'pilot'        # 'pilot' | 'full'
PILOT_EXTRA = 150     # số study không nhãn thêm vào pilot (toàn bộ gold luôn được giữ)
PILOT_EPOCHS = 3
assert MODE in ('pilot', 'full')

In [ ]:
# 2. Đăng nhập HuggingFace bằng Colab Secret 'HF_TOKEN'
#    Cần đã bấm "Agree" cho google/gemma-3-12b-it (model gated) bằng đúng tài khoản này.
import os
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN
print('Đã đăng nhập HuggingFace.')

## Cấu hình

Cấu hình là dataclass bất biến nạp từ YAML. Ghi đè ở đây **có tác dụng thật** vì
`cfg` được truyền tường minh xuống mọi hàm — khác với cách dùng hằng số cấp
module, nơi `from module import CONSTANT` sao chép giá trị lúc import và mọi ghi
đè sau đó bị nuốt im lặng.

Pilot ghi artifacts vào `artifacts/pilot/` (tách hẳn checkpoint, mask, cache) để
`resume=True` của bản full không vô tình nối tiếp checkpoint của pilot.

In [ ]:
# 3. Nạp cấu hình
import os
from knee_mri.config import load_config
from knee_mri.utils.logging import setup_logging
from knee_mri.utils.seed import set_seed
import torch

BASE_ARTIFACTS = DRIVE_ROOT / 'artifacts'
ARTIFACTS = BASE_ARTIFACTS / ('pilot' if MODE == 'pilot' else '')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

overrides = {'paths': {
    'data_root': str(DRIVE_ROOT / 'dataset'),
    'artifacts_root': str(ARTIFACTS),
    'dino_code_dir': str(DRIVE_ROOT / 'vendor' / '3DINO'),
}}
if MODE == 'pilot':
    overrides['train'] = {'epochs': PILOT_EPOCHS}

cfg = load_config('colab', overrides=overrides)
cfg.paths.ensure()
setup_logging(logging.INFO, log_dir=cfg.paths.logs)
set_seed(cfg.seed)

print('MODE         :', MODE)
print('dataset      :', cfg.paths.data_root)
print('artifacts    :', cfg.paths.artifacts_root)
print('target_shape :', cfg.data.target_shape)
print('VLM          :', cfg.vlm.model_id, '| epochs:', cfg.train.epochs)
print('thiết bị     :', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 4. Clone 3DINO (chứa package `dinov2`) và tải weights — URL, tên file, thư mục lấy từ cfg.
#    License CC BY-NC-ND: clone nguyên gốc, KHÔNG sửa code trong vendor/.
#    requirements của 3DINO được cài với *constraints* khóa torch/transformers/numpy ở
#    đúng bản Colab đang có, để chúng không bị hạ/nâng cấp âm thầm.
import glob, shutil
from huggingface_hub import snapshot_download

REPO = cfg.paths.dino_code_dir
if not REPO.is_dir():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', cfg.dino.repo_url, str(REPO)], check=True)

PROTECTED = {'torch', 'torchvision', 'torchaudio', 'transformers', 'tokenizers', 'numpy',
             'huggingface-hub', 'accelerate', 'safetensors'}
def frozen_versions():
    out = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
    return dict(l.split('==', 1) for l in out.splitlines() if '==' in l)
before = {k: v for k, v in frozen_versions().items() if k.lower() in PROTECTED}
constraints = Path('/content/constraints.txt')
constraints.write_text('\n'.join(f'{k}=={v}' for k, v in before.items()))

pip = [sys.executable, '-m', 'pip', 'install', '-q']
result = subprocess.run(pip + ['-c', str(constraints), '-r', str(REPO / 'requirements.txt')])
if result.returncode != 0:
    print('! requirements 3DINO xung đột với bản torch/transformers của Colab -> cài --no-deps.')
    print('  Nếu import dinov2 bên dưới báo thiếu module, cài thủ công module đó.')
    subprocess.run(pip + ['--no-deps', '-r', str(REPO / 'requirements.txt')])

# Weights luôn nằm ở artifacts gốc; pilot dùng symlink để khỏi chép ~1GB.
dino_weights = cfg.dino.weights_path(BASE_ARTIFACTS)
if not dino_weights.exists():
    downloaded = snapshot_download('AICONSlab/3DINO-ViT', token=HF_TOKEN)
    shutil.copy(glob.glob(os.path.join(downloaded, cfg.dino.weight_file))[0], dino_weights)
pilot_link = cfg.dino.weights_path(cfg.paths.artifacts_root)
if not pilot_link.exists():
    os.symlink(dino_weights, pilot_link)

# Kiểm tra torch/transformers không bị đổi và dinov2 import được
after = frozen_versions()
changed = {k: (v, after.get(k)) for k, v in before.items() if after.get(k) != v}
assert not changed, f'Dependency bị đổi bởi cài 3DINO: {changed} — Restart runtime và kiểm tra lại.'
sys.path.insert(0, str(REPO))
from dinov2.configs import load_and_merge_config_3d  # noqa: F401  (smoke test import)
print('3DINO code   :', REPO)
print('3DINO weights:', pilot_link)

In [ ]:
# 5. Nạp catalog (index CSV MỘT LẦN, tra cứu O(1) sau đó) và chọn tập study
from knee_mri.data.catalog import StudyCatalog
from knee_mri.data.splits import pilot_subset

catalog = StudyCatalog.from_csv(cfg.paths.train_csv, cfg.paths.train_series_csv)
all_studies = catalog.studies_with_series()
n_gold = len(catalog.studies_with_gold())

# Pilot = TOÀN BỘ gold (làm validation) + mẫu ngẫu nhiên cố định seed. Không cắt 50 study
# đầu như trước — cách đó chỉ giữ lại 1 study gold.
subset = pilot_subset(catalog, extra=PILOT_EXTRA, seed=cfg.seed) if MODE == 'pilot' else all_studies

print(catalog)
print(f'MODE={MODE}: {len(subset)} study xử lý ({n_gold} gold làm validation)')
assert n_gold >= 10, 'Quá ít study gold có series — không thể validation.'

## Kiểm tra trước khi chạy (preflight)

Chặn lỗi rẻ-mà-đắt: token HF chưa được cấp quyền, `guidance_dim` lệch với VLM thật,
thiếu GPU, hết dung lượng Drive. Tất cả đều dừng *trước* khi tốn giờ GPU.

In [ ]:
# 6. Preflight — dừng ở đây nếu có vấn đề, trước khi tốn giờ GPU
from knee_mri.pipeline import run_preflight

preflight = run_preflight(cfg, catalog, subset, hf_token=HF_TOKEN)
print(preflight.info)
preflight.raise_if_failed()
print('Preflight OK.')

In [ ]:
# 7. (Tùy chọn) So sánh các chế độ chuẩn hóa trên mẫu nhỏ — chỉ chạy ở pilot
RUN_NORM_ANALYSIS = (MODE == 'pilot')
if RUN_NORM_ANALYSIS:
    out = subprocess.run(
        [sys.executable, str(CODE / 'scripts' / 'analyze_normalization.py'),
         '--env', 'colab', '--max-series', '60'],
        capture_output=True, text=True, cwd=str(CODE),
    )
    print(out.stdout[-3000:] or out.stderr[-3000:])

In [ ]:
# 8. Đo thời gian từng bước + tự thử lại khi lỗi thoáng qua (OOM tạm thời, mất kết nối Drive).
#    Mọi bước precompute bỏ qua file đã có, nên thử lại không làm lại phần đã xong.
#    Lỗi cấu hình/thiếu asset (ImportError, thiếu 3DINO...) thì dừng ngay.
from knee_mri.pipeline import StageTimer

timer = StageTimer()

## S1 — Mask ROI và weak label

In [ ]:
# 9. Sinh mask ROI bằng SAM
from knee_mri.pipeline import generate_masks
timer.run('masks', generate_masks, cfg, catalog, subset, device='cuda')

In [ ]:
# 10. Sinh weak label từ report
#     use_vlm=True dùng VLM đọc hiểu (lỗi từng study tự lùi về luật từ khóa).
from knee_mri.pipeline import precompute_weak_labels
timer.run('weak_labels', precompute_weak_labels, cfg, catalog, subset, use_vlm=True, device='cuda')

## S2 — Precompute teacher rồi huấn luyện student

In [ ]:
# 11. Đặc trưng ảnh từ 3DINO-ViT (teacher đóng băng → cache một lần)
from knee_mri.pipeline import precompute_teacher_features
timer.run('teacher_feats', precompute_teacher_features, cfg, catalog, subset, device='cuda')

In [ ]:
# 12. Guidance từ VLM đa modality (ảnh + report → G)
#     KHÔNG bật allow_zero_fallback: guidance toàn 0 làm loss contrastive vô nghĩa.
#     Lỗi thoáng qua được thử lại; lỗi lặp lại thì dừng để bạn xử lý.
from knee_mri.pipeline import precompute_guidance
timer.run('guidance', precompute_guidance, cfg, catalog, subset, device='cuda')

In [ ]:
# 13. Kiểm tra cache trước khi huấn luyện — thiếu guidance/teacher sẽ bị thay bằng vector 0
#     mà không báo lỗi, nên chặn ở đây.
from knee_mri.data.dataset import describe_cache, missing_cache

print(describe_cache(cfg, subset))
gaps = missing_cache(cfg, subset)
assert not gaps, f'Thiếu cache (số study): {gaps}. Chạy lại các cell 9-12.'

In [ ]:
# 14. Chia tập: study gold = validation cố định, phần còn lại (trong subset) = train
from knee_mri.data.dataset import KneeDataset
from knee_mri.data.splits import make_split
from knee_mri.training.trainer import train

split = make_split(catalog, seed=cfg.seed, subset=subset)
print(split)
assert len(split.val) >= 10, f'Validation chỉ có {len(split.val)} study — AUC không có nghĩa.'

train_ds = KneeDataset(split.train, catalog, cfg)
val_ds = KneeDataset(split.val, catalog, cfg)

In [ ]:
# 15. Huấn luyện student. resume=True: nếu runtime bị ngắt, chạy lại cell này (hoặc cả
#     notebook) để tiếp tục từ artifacts/<...>/checkpoints/last.pt trên Drive.
result = train(cfg, train_ds, val_ds, device='cuda', resume=True)
print('Macro AUC tốt nhất:', result.best_metric)
print('Checkpoint        :', result.best_checkpoint)
epoch_seconds = (
    sum(r['seconds'] for r in result.history) / len(result.history) if result.history else None
)

## Đánh giá độc lập

AUC bên dưới tính lại từ checkpoint tốt nhất bằng đường **inference thật**
(`Predictor`, chỉ ảnh + metadata) trên study gold — không dùng cache teacher/guidance,
nên là con số trung thực nhất về chất lượng lúc nộp bài.

In [ ]:
# 16. Đánh giá checkpoint tốt nhất trên các study gold
from knee_mri.inference.predictor import Predictor, evaluate_gold
from knee_mri.training.metrics import format_auc_report

predictor = Predictor.from_checkpoint(cfg, checkpoint=result.best_checkpoint, device='cuda')
evaluation = evaluate_gold(predictor, catalog, split.val, log_every=0)
print(f'{len(evaluation.study_uids)} study gold')
print(format_auc_report(evaluation.report))
if evaluation.report.macro < 0.55:
    print('! Macro AUC gần ngẫu nhiên — bình thường với pilot rất ngắn, nhưng nếu đã chạy '
          'full thì kiểm tra loss, cache guidance/teacher và nhãn yếu trước khi nộp.')
del predictor

## S3 — Giải thích và sinh submission

In [ ]:
# 17. CAM + báo cáo, đối chiếu với mask S1 — ưu tiên study gold (đã chắc chắn có mask)
from knee_mri.pipeline import explain_studies

explain_uids = (split.val + split.train)[:20]
results = explain_studies(cfg, catalog, explain_uids, use_vlm=True, device='cuda')
for item in results[:3]:
    print(item.study_uid[:24], '| IoU CAM/ROI:', item.overlap)

In [ ]:
# 18. Sinh submission.csv (CHỈ ảnh + metadata). Chỉ chạy ở MODE='full' — pilot chưa đủ
#     tốt để nộp và dự đoán cả tập test rất tốn thời gian.
if MODE == 'full':
    from knee_mri.inference import predict_submission
    import pandas as pd
    submission_path = predict_submission(cfg, device='cuda')
    print('Submission:', submission_path)
    display(pd.read_csv(submission_path).head())
else:
    print("MODE='pilot' → bỏ qua. Đổi sang 'full', Restart runtime và chạy lại để nộp bài.")

## Ước lượng cho bản đầy đủ

Chạy sau pilot (khi `timer.seconds` vừa được đo từ đầu — nếu một bước bị bỏ qua vì đã
có cache thì thời gian của nó thấp giả tạo).

In [ ]:
# 19. Ngoại suy thời gian từ pilot cho toàn bộ dataset
from knee_mri.pipeline import estimate_full_run

FULL_EPOCHS = 30
hours = estimate_full_run(
    timer.seconds,
    pilot_studies=len(subset),
    full_studies=len(all_studies),
    epoch_seconds=epoch_seconds,
    pilot_train=len(split.train),
    full_train=len(all_studies) - n_gold,
    full_epochs=FULL_EPOCHS,
)
print(f'Pilot: {len(subset)} study | Full: {len(all_studies)} study')
for stage, value in hours.items():
    if stage != 'total':
        print(f'  {stage:<14s} ≈ {value:5.1f} giờ')
print(f'Tổng ≈ {hours["total"]:.1f} giờ GPU A100 (ước lượng thô; chia thành nhiều phiên được vì có resume).')

## Tiếp theo

Mở [`03_results_visualization.ipynb`](03_results_visualization.ipynb) để xem
phân bố dự đoán, CAM overlay và đường ROC trên tập nhãn gold (đặt `RUN` trùng `MODE`).